#Day 2 : Data Cleaning + SQL Database Design


In [15]:
# Data Cleaning and Preprocessing for NAV History File

import pandas as pd
import numpy as np

# 1. Load the data
df_nav = pd.read_csv(r"../data/processed/02_nav_history.csv")
# 2. Parse dates to datetime format
df_nav['date'] = pd.to_datetime(df_nav['date'])
# 3. Sort by amfi_code and date (important for forward-filling)
df_nav = df_nav.sort_values(by=['amfi_code', 'date'])
# 4. Remove duplicate rows
df_nav = df_nav.drop_duplicates()
# 5. Validate NAV > 0 (Replace 0 or negative values with NaN, then forward fill)
df_nav.loc[df_nav['nav'] <= 0, 'nav'] = np.nan
# 6. Forward-fill missing NAV values within each fund group (amfi_code)
df_nav['nav'] = df_nav.groupby('amfi_code')['nav'].ffill()
# Save the cleaned file
df_nav.to_csv('../data/processed/nav_history_cleaned.csv', index=False)
print("nav_history.csv cleaned and saved!")


nav_history.csv cleaned and saved!


In [252]:
df_invester_transactions = pd.read_csv(r"../data/processed/08_investor_transactions.csv")   
print(df_invester_transactions.head())

  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verified  
2      M

In [16]:
# Data Cleaning and Preprocessing for investor_transactions
import pandas as pd
import numpy as np
# 1. Load the data
df_trans = pd.read_csv('../data/raw/08_investor_transactions.csv')
# 2. Fix date formats
df_trans['transaction_date'] = pd.to_datetime(df_trans['transaction_date'])
# 3. Standardize transaction_type values (e.g., SIP, Lumpsum, Redemption)
# This maps messy text to standardized strings
trans_map = {
    'sip': 'SIP',
    'lumpsum': 'Lumpsum',
    'redemption': 'Redemption'
}
df_trans['transaction_type'] = df_trans['transaction_type'].replace(trans_map)
# 4. Validate amount > 0 (Keep only positive amounts)
df_trans = df_trans[df_trans['amount_inr'] > 0]
# 5. Check KYC status enum values (Ensure they are standard, e.g., 'Yes' or 'No')
df_trans['kyc_status'] = df_trans['kyc_status'].str.capitalize() # Converts 'yes'/'YES' to 'Yes'

# Save the cleaned file
df_trans.to_csv('../data/processed/investor_transactions_cleaned.csv', index=False)
print("investor_transactions.csv cleaned and saved!")
print(df_trans.head())

investor_transactions.csv cleaned and saved!
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Ve

In [17]:
# Data Cleaning and Preprocessing for scheme performance
import pandas as pd
import numpy as np
# 1. Load the data
df_perf = pd.read_csv('../data/raw/07_scheme_performance.csv')
# 2. Validate return values are numeric (turn errors into NaN)
cols_to_convert = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
df_perf[cols_to_convert] = df_perf[cols_to_convert].apply(pd.to_numeric, errors='coerce')
# 3. Flag anomalies (e.g., return values greater than 100% or less than -50%)
# 3. Flag anomalies (returns greater than 100% or less than -50% in ANY of the performance columns)
df_perf['is_anomaly'] = (
    (df_perf['return_1yr_pct'] > 100) | (df_perf['return_1yr_pct'] < -50) |
    (df_perf['return_3yr_pct'] > 100) | (df_perf['return_3yr_pct'] < -50) |
    (df_perf['return_5yr_pct'] > 100) | (df_perf['return_5yr_pct'] < -50)
)
# 4. Filter or flag expense_ratio range (0.1% to 2.5%, assuming expressed as percentages)
# If it's outside this range, we can cap it or flag it. Let's filter it to be safe.
df_perf = df_perf[(df_perf['expense_ratio_pct'] >= 0.1) & (df_perf['expense_ratio_pct'] <= 2.5)]
# Save the cleaned file

df_perf.to_csv('../data/processed/scheme_performance_cleaned.csv', index=False)
print("scheme_performance.csv cleaned and saved!")

scheme_performance.csv cleaned and saved!


In [26]:
#AUM File Processing

import pandas as pd
import numpy as np
import sqlite3

# Load the CSV file from your spreadsheet
df_aum_raw = pd.read_csv('C:\\Users\\hp\\Desktop\\mutual_fund_project\\data\\processed\\03_aum_by_fund_house.csv')
df_aum_raw['date'] = pd.to_datetime(df_aum_raw['date'])
df_aum_raw = df_aum_raw.drop_duplicates()
df_aum_raw.to_csv('../data/processed/aum_cleaned.csv', index=False)

print("Cleaning Done")

Cleaning Done


In [19]:
#SQLite Database Design & Loading
import sqlite3
conn = sqlite3.connect('../data/processed/bluestock_mf.db')
cursor=conn.cursor()

schema_statements = """
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_name TEXT,
    category TEXT
);

CREATE TABLE IF NOT EXISTS dim_date (
    date TEXT PRIMARY KEY,
    day INTEGER,
    month INTEGER,
    year INTEGER,
    quarter INTEGER
);

CREATE TABLE IF NOT EXISTS fact_nav (
    NAV_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);

-- Table: fact_transactions (Tracks investor purchase and redemption behaviors)
CREATE TABLE fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    transaction_date TEXT,
    transaction_type TEXT NOT NULL, -- e.g., 'SIP', 'Lumpsum', 'Redemption'
    amount REAL NOT NULL,
    kyc_status TEXT,
    state TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund (amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date (date)
);

-- Table: fact_performance (Tracks operational metrics and fund yields)
CREATE TABLE fact_performance (
    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    returns REAL,
    expense_ratio REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund (amfi_code)
);
CREATE TABLE fact_aum (
    aum_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT,
    fund_house TEXT,
    aum_lakh_crore REAL,
    aum_crore REAL,
    num_schemes INTEGER
);
"""
# Execute the table creation statements
cursor.executescript(schema_statements)
print("Database schema tables created successfully!")

Database schema tables created successfully!


In [4]:
#SQLite Database Design & Loading
import sqlite3
conn = sqlite3.connect('../data/processed/bluestock_mf.db')
cursor=conn.cursor()

schema_statements = """
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_name TEXT,
    category TEXT
);

CREATE TABLE IF NOT EXISTS dim_date (
    date TEXT PRIMARY KEY,
    day INTEGER,
    month INTEGER,
    year INTEGER,
    quarter INTEGER
);

CREATE TABLE IF NOT EXISTS fact_nav (
    NAV_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date)
);

-- Table: fact_transactions (Tracks investor purchase and redemption behaviors)
CREATE TABLE fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    transaction_date TEXT,
    transaction_type TEXT NOT NULL, -- e.g., 'SIP', 'Lumpsum', 'Redemption'
    amount REAL NOT NULL,
    kyc_status TEXT,
    state TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund (amfi_code),
    FOREIGN KEY (transaction_date) REFERENCES dim_date (date)
);

-- Table: fact_performance (Tracks operational metrics and fund yields)
CREATE TABLE fact_performance (
    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    returns REAL,
    expense_ratio REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund (amfi_code)
);
CREATE TABLE fact_aum (
    aum_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT,
    fund_house TEXT,
    aum_lakh_crore REAL,
    aum_crore REAL,
    num_schemes INTEGER
);
"""
# Execute the table creation statements
cursor.executescript(schema_statements)
print("Database schema tables created successfully!")

Database schema tables created successfully!


In [30]:
import sqlite3

conn = sqlite3.connect('../data/processed/bluestock_mf.db')

# Change 'append' to 'replace' to overwrite the table schema
df_nav.to_sql('fact_nav', conn, if_exists='replace', index=False)
df_trans.to_sql('fact_transactions', conn, if_exists='replace', index=False)
df_perf.to_sql('fact_performance', conn, if_exists='replace', index=False)
df_aum_raw.to_sql('fact_aum', conn, if_exists='replace', index=False)
print("Data Transferred Succesfully")
conn.commit()
conn.close()

Data Transferred Succesfully


In [31]:
# Row Count Verification Check 
# to perform a quick audit to make sure no data was lost, corrupted, or accidentally duplicated when we transferred it from 
# Python environment into your SQLite database.
import sqlite3
conn = sqlite3.connect('../data/processed/bluestock_mf.db')
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM fact_nav")

print(f"Total rows in fact_nav: {cursor.fetchone()[0]}")
print(f"Total rows in fact_transactions: {cursor.execute('SELECT COUNT(*) FROM fact_transactions').fetchone()[0]}")
print(f"Total rows in fact_performance: {cursor.execute('SELECT COUNT(*) FROM fact_performance').fetchone()[0]}")
print(f"Total rows in fact_aum:{cursor.execute('SELECT COUNT(*) FROM fact_aum').fetchone()[0]} ")
print(f"Total rows in original cleaned dataframe: {len(df_nav)}")
print(f"Total rows in original cleaned dataframe: {len(df_trans)}")
print(f"Total rows in original cleaned dataframe: {len(df_perf)}")
print(f"Total rows in original cleaned dataframe: {len(df_aum_raw)}")

# 3. Safely close it at the very end
conn.close()

Total rows in fact_nav: 46000
Total rows in fact_transactions: 32778
Total rows in fact_performance: 40
Total rows in fact_aum:90 
Total rows in original cleaned dataframe: 46000
Total rows in original cleaned dataframe: 32778
Total rows in original cleaned dataframe: 40
Total rows in original cleaned dataframe: 90


In [12]:
import sqlite3

# 1. Re-open the connection and cursor
conn = sqlite3.connect('../data/processed/bluestock_mf.db')
cursor = conn.cursor()

# 2. Clear out the incorrect table
cursor.execute("DROP TABLE IF EXISTS fact_nav;")
conn.commit()

# 3. Re-import the correct dataframe using pandas
df_nav.to_sql('fact_nav', conn, if_exists='replace', index=False)
print("fact_nav re-uploaded successfully!")

# 4. (Optional) Run your verification right here to see if it matches now
cursor.execute("SELECT COUNT(*) FROM fact_nav")
print(f"New count in fact_nav table: {cursor.fetchone()[0]}")
print(f"Count in df_nav dataframe:   {len(df_nav)}")

# 5. Safely close it when you are completely finished
conn.close()

fact_nav re-uploaded successfully!
New count in fact_nav table: 46000
Count in df_nav dataframe:   46000


In [24]:
# 1. Open Connection
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/processed/bluestock_mf.db')
cursor = conn.cursor()

# 2. Drop the bloated tables to clear them out
cursor.execute("DROP TABLE IF EXISTS fact_nav;")
conn.commit()

In [14]:
# 1. Open Connection
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/processed/bluestock_mf.db')
cursor = conn.cursor()

# 2. Drop the bloated tables to clear them out
cursor.execute("DROP TABLE IF EXISTS fact_nav;")
cursor.execute("DROP TABLE IF EXISTS fact_transactions;")
cursor.execute("DROP TABLE IF EXISTS fact_performance;")
cursor.execute("DROP TABLE IF EXISTS fact_aum;")
conn.commit()

In [ ]:
#Connect to SQLite database
conn = sqlite3.connect('../data/processed/mutual_funds.db')
cursor = conn.cursor()
# Drop the old table if it doesn't match our spreadsheet columns
cursor.execute("DROP TABLE IF EXISTS fact_aum;")

# Create the updated table matching the data columns precisely
cursor.execute("""
CREATE TABLE fact_aum (
    aum_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT,
    fund_house TEXT,
    aum_lakh_crore REAL,
    aum_crore REAL,
    num_schemes INTEGER
);
""")
print("Updated fact_aum schema table created successfully!")

Updated fact_aum schema table created successfully!


In [ ]:
# Select and reorder columns to match our SQL schema structure
df_aum_raw = df_aum_raw[['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']]

# Load the data frame into SQLite
df_aum_raw.to_sql('fact_aum', conn, if_exists='append', index=False)
df_nav.to_sql('fact_nav', conn, if_exists='append', index=False)
print("AUM data successfully loaded into fact_aum table!")

# Row Count Verification Check
cursor.execute("SELECT COUNT(*) FROM fact_aum")
db_count = cursor.fetchone()[0]
df_count = len(df_aum_raw)

print("--- Verification Check ---")
print(f"Rows inside original DataFrame: {df_count}")
print(f"Rows written to SQLite Table:   {db_count}")

# Close connection
conn.close()

AUM data successfully loaded into fact_aum table!
--- Verification Check ---
Rows inside original DataFrame: 90
Rows written to SQLite Table:   90


In [256]:
import sqlite3

# 1. Connect to your SQLite database
conn = sqlite3.connect('../data/processed/mutual_funds.db')  # Replace with your DB path
cursor = conn.cursor()

# 2. Execute the UPDATE statement
# This handles extra spaces, case-insensitivity, and leaves other values untouched
update_query = """
UPDATE fact_transactions
SET transaction_type = CASE 
    WHEN TRIM(LOWER(transaction_type)) = 'sip' THEN 'SIP'
    WHEN TRIM(LOWER(transaction_type)) = 'lumpsum' THEN 'Lumpsum'
    WHEN TRIM(LOWER(transaction_type)) = 'redemption' THEN 'Redemption'
    ELSE 'Unknown' -- Optional: Converts existing NaNs or text anomalies to 'Unknown'
END;
"""

try:
    cursor.execute(update_query)
    conn.commit()
    print(f"Successfully updated {cursor.rowcount} rows in fact_transactions!")
except Exception as e:
    conn.rollback()
    print(f"An error occurred: {e}")
finally:
    # 3. Close the connection
    conn.close()

Successfully updated 32778 rows in fact_transactions!


In [257]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/processed/mutual_funds.db') # Use your exact DB file name
df_check = pd.read_sql_query("SELECT DISTINCT transaction_type FROM fact_transactions;", conn)
print(df_check)
conn.close()

  transaction_type
0          Unknown


In [263]:
import sqlite3

# 1. Connect to your database path
conn = sqlite3.connect('../data/processed/mutual_funds.db')

# 2. Overwrite the table with your clean DataFrame (if_exists='replace' ensures no duplication)
df_trans.to_sql('fact_transactions', conn, if_exists='replace', index=False)

conn.close()
print("Database table successfully overwritten with your clean DataFrame!")

Database table successfully overwritten with your clean DataFrame!


In [32]:
# 1. Stage the files explicitly
git add data/processed/ bluestock_mf.db schema.sql queries.sql data_dictionary.md

# 2. Commit with the required milestone string
git commit -m "Day 2: Cleaned data + SQLite DB loaded"

SyntaxError: invalid syntax (2758922555.py, line 2)